# Pola Arsitektur Convnet Modern dan Penyetelan Hiperparameter

**Buku:** *Deep Learning dengan Python* — Bab 7, Praktik Terbaik Lanjutan

Membandingkan convnet polos dengan convnet yang memakai *batch normalization*, koneksi residual, dan konvolusi *depthwise separable*. Lalu mencoba presisi campuran dan pencarian hiperparameter otomatis dengan KerasTuner.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Kode bersifat *backend-agnostik* (TensorFlow, JAX, atau PyTorch).

**Perkiraan waktu (GPU T4):** ±20 menit

**Isi notebook:**
- Baseline convnet polos pada CIFAR-10.
- Blok residual + BatchNorm + `SeparableConv2D` (pola ala Xception).
- Presisi campuran (`mixed_float16`) untuk mempercepat di GPU.
- KerasTuner: `BayesianOptimization` untuk memilih jumlah unit dan optimizer.

In [ ]:
!pip install -q keras-tuner

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Data CIFAR-10

In [ ]:
import numpy as np
from keras import layers
import matplotlib.pyplot as plt

(x_latih, y_latih), (x_uji, y_uji) = keras.datasets.cifar10.load_data()
x_latih, x_val = x_latih[5000:], x_latih[:5000]
y_latih, y_val = y_latih[5000:], y_latih[:5000]
print(x_latih.shape, x_val.shape, x_uji.shape)

augmentasi = keras.Sequential([layers.RandomFlip("horizontal"),
                               layers.RandomTranslation(0.1, 0.1)], name="augmentasi")

def latih(model, judul, epochs=30):
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    r = model.fit(x_latih, y_latih, epochs=epochs, batch_size=128, validation_data=(x_val, y_val))
    plt.plot(r.history["val_accuracy"], label=judul)
    print(f"{judul}: akurasi uji {model.evaluate(x_uji, y_uji, verbose=0)[1]:.2%} | parameter {model.count_params():,}")
    return r

## 2. Baseline: convnet polos

In [ ]:
def convnet_polos():
    masukan = keras.Input(shape=(32, 32, 3))
    x = augmentasi(masukan)
    x = layers.Rescaling(1 / 255)(x)
    for f in [32, 64, 128]:
        x = layers.Conv2D(f, 3, activation="relu", padding="same")(x)
        x = layers.Conv2D(f, 3, activation="relu", padding="same")(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.5)(x)
    return keras.Model(masukan, layers.Dense(10, activation="softmax")(x))

plt.figure(figsize=(8, 4))
r_polos = latih(convnet_polos(), "polos")

## 3. Residual + BatchNorm + SeparableConv
Pola: `BN → ReLU → SeparableConv`, dua kali, lalu tambahkan jalan pintas (*shortcut*). Bila jumlah filter berubah, jalan pintas memakai konvolusi 1×1.

In [ ]:
def blok_residual(x, filter):
    residu = x
    for _ in range(2):
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(filter, 3, padding="same", use_bias=False)(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)
    residu = layers.Conv2D(filter, 1, strides=2, padding="same", use_bias=False)(residu)
    return layers.add([x, residu])

def convnet_modern(dtype_keluaran=None):
    masukan = keras.Input(shape=(32, 32, 3))
    x = augmentasi(masukan)
    x = layers.Rescaling(1 / 255)(x)
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)   # konvolusi biasa di awal
    for f in [64, 128, 256]:
        x = blok_residual(x, f)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    return keras.Model(masukan, layers.Dense(10, activation="softmax", dtype=dtype_keluaran)(x))

r_modern = latih(convnet_modern(), "residual+BN+separable")
plt.xlabel("Epoch"); plt.ylabel("Akurasi validasi"); plt.legend(); plt.show()

## 4. Presisi campuran
Perhitungan dalam `float16`, bobot tetap `float32`. Lapisan keluaran (softmax) dipaksa `float32` agar stabil secara numerik.

In [ ]:
import time

keras.mixed_precision.set_global_policy("mixed_float16")
m = convnet_modern(dtype_keluaran="float32")
m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
t0 = time.time()
m.fit(x_latih, y_latih, epochs=3, batch_size=128, validation_data=(x_val, y_val))
print(f"3 epoch presisi campuran: {time.time() - t0:.0f} detik")
keras.mixed_precision.set_global_policy("float32")      # kembalikan ke default

## 5. KerasTuner: pencarian hiperparameter
Contoh kecil pada MNIST agar cepat. Pola yang sama berlaku untuk model besar.

In [ ]:
import keras_tuner as kt

(xm, ym), (xm_uji, ym_uji) = keras.datasets.mnist.load_data()
xm = xm.reshape(-1, 784).astype("float32") / 255
xm_uji = xm_uji.reshape(-1, 784).astype("float32") / 255

def bangun_model(hp):
    unit = hp.Int("unit", min_value=16, max_value=128, step=16)
    optimizer = hp.Choice("optimizer", ["rmsprop", "adam"])
    model = keras.Sequential([keras.Input(shape=(784,)),
                              layers.Dense(unit, activation="relu"),
                              layers.Dense(10, activation="softmax")])
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

MAKS_PERCOBAAN = 10
tuner = kt.BayesianOptimization(bangun_model, objective="val_accuracy",
                                max_trials=MAKS_PERCOBAAN, directory="tuner_mnist", overwrite=True)
tuner.search(xm, ym, batch_size=128, epochs=10, validation_split=0.2,
             callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3)])
terbaik = tuner.get_best_hyperparameters(1)[0]
print("Hiperparameter terbaik:", terbaik.values)

In [ ]:
# Latih ulang konfigurasi terbaik pada seluruh data latih, lalu uji sekali saja
model = bangun_model(terbaik)
model.fit(xm, ym, batch_size=128, epochs=10, verbose=0)
print(f"Akurasi uji: {model.evaluate(xm_uji, ym_uji, verbose=0)[1]:.2%}")

**Ingat:** pencarian hiperparameter juga bisa *overfit* ke data validasi. Data uji hanya disentuh di langkah terakhir.